# Week 11 · Day 4 — Your First QLoRA Fine-Tune

**Goal today:** get a **working end-to-end training pipeline** — load the base model, attach small LoRA "sticky notes", train them on your data, and save the result. Today we care about *"does the whole thing run and produce a sensible model?"* — **not** about squeezing out the best score (that's Day 5's experiments).

> **Keep the GPU ON** (Settings → Accelerator → GPU) and **Internet ON**.

## Where we are
- Day 1: setup + inspection. Day 2: clean/dedupe/split + benchmark. Day 3: measured the **base model** (your "before" score).
- **Day 4 (today):** fine-tune with QLoRA and save the adapter.

## How to run on Kaggle
1. **+ Add Input** → attach the dataset with your `train.jsonl` / `val.jsonl` (from Day 2). *If you didn't save those, this notebook rebuilds them from your combined data automatically.*
2. GPU + Internet **ON**.
3. Run top to bottom. The first run uses a **short 60-step test** so you can confirm it works quickly; flip one switch for the full run.
4. **Save Version** at the end to keep the adapter.

## The ideas you'll learn today

- **SFT (supervised fine-tuning)** — we show the model pairs of *(question → good answer)* and nudge it to produce answers like those. That's all "training" means here.
- **LoRA adapters** — we **freeze** the giant model and train only tiny add-on matrices (a few tens of MB). Cheap, fast, and the thing you publish.
- **The LoRA knobs** — `r` (size/capacity), `alpha` (strength), `dropout` (randomly ignore some notes to avoid cramming). We start at r=16, alpha=16, dropout=0.05 — sensible defaults, not final answers.
- **Completion-only training** — we compute the learning signal **only on the assistant's answer**, not on the question. Otherwise the model wastes effort learning to *repeat questions*.
- **Epochs & overfitting** — an *epoch* is one pass over the data. Too many passes → the model **memorizes** instead of **learning**. We watch the **validation loss**: if it rises while training loss keeps falling, that's overfitting.
- **Why Qwen2.5-7B-Instruct** — strong at math/technical reasoning for its size, and Apache-2.0 licensed (clean to publish). It already knows a lot; we're just adapting its style to your domain.

In [ ]:
# ---- Day 4 needs a GPU. Kaggle: Settings -> Accelerator -> GPU (T4 x2 or P100), Internet -> ON ----
# We are using the standard Hugging Face QLoRA stack instead of Unsloth.
# T4-safe setup: force ONE visible GPU and reduce CUDA memory fragmentation.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # critical: avoid DataParallel + 4-bit instability
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U "transformers>=4.55,<5" "peft>=0.18,<0.20" "trl>=0.20,<0.25" "accelerate>=1.0,<2" "bitsandbytes>=0.45,<0.48" "datasets>=3.4.1,<4.4" sentencepiece

import torch, transformers, peft, trl, datasets
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("trl         :", trl.__version__)
print("datasets    :", datasets.__version__)

if torch.cuda.is_available():
    print("Visible CUDA devices:", torch.cuda.device_count(), "(should be 1)")
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported(), "(T4/P100 usually False -> fp16)")
else:
    print("NO GPU DETECTED! Turn it on: Settings -> Accelerator -> GPU, then re-run.")

## Step 1 — Load (or rebuild) your training data

**What:** load `train.jsonl` and `val.jsonl` from Day 2. Each row is a chat conversation: `system → user (question) → assistant (answer)`.

**Why:** *train* is what the model studies; *val* is a held-out set we watch to catch overfitting.

**Safety net:** Kaggle deletes `/kaggle/working/` when a session ends, so if your Day-2 files aren't attached, this cell **rebuilds** clean, deduped splits from your combined data automatically (same logic as Day 2).

**What could go wrong:**
- *Nothing found* → attach your Day-2 output (or the `all_combined.jsonl`) with **+ Add Input**.

**How we check:** it prints the train/val sizes and shows one formatted example.

In [ ]:
import json, glob, os, re, random
random.seed(3407)

def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

def find(*names):
    hits = []
    for n in names:
        hits += glob.glob(f"/kaggle/input/**/{n}", recursive=True)
        hits += glob.glob(f"/kaggle/working/**/{n}", recursive=True)
        hits += glob.glob(f"data/**/{n}", recursive=True)
        if os.path.exists(n):
            hits.append(n)
    return hits

tr, va = find("train.jsonl"), find("val.jsonl")
if tr and va:
    train_rows, val_rows = read_jsonl(tr[0]), read_jsonl(va[0])
    print(f"Loaded ready-made splits:\n  {tr[0]} ({len(train_rows)})\n  {va[0]} ({len(val_rows)})")
else:
    print("No train.jsonl/val.jsonl found -> rebuilding splits from source data (Day-2 logic).")
    SYSTEM_PROMPT = ("You are an expert RF, DSP, and wireless communications engineer. "
                     "Answer precisely and show the key steps. When code is required, "
                     "return correct, runnable Python.")

    def build_answer(r):
        t = r.get("task_type", "")
        if t == "code":
            o = (r.get("output") or "").strip()
            return f"```python\n{o}\n```" if o else ""
        if t == "numeric":
            p = []
            if (r.get("reasoning") or "").strip():
                p.append(r["reasoning"].strip())
            if r.get("final_answer") is not None:
                u = (r.get("unit") or "").strip()
                p.append(f"**Final answer:** {r['final_answer']}{(' ' + u) if u else ''}".rstrip())
            if (r.get("solver_code") or "").strip():
                p.append(f"```python\n{r['solver_code'].strip()}\n```")
            return "\n\n".join(p)
        return (r.get("output") or r.get("solver_code") or "").strip()

    def to_messages(r):
        u = (r.get("instruction") or "").strip()
        if (r.get("input") or "").strip():
            u += "\n\nInput:\n" + r["input"].strip()
        return [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": u},
                {"role": "assistant", "content": build_answer(r)}]

    def norm(t):
        return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9\s]", " ", (t or "").lower())).strip()

    src = find("all_combined.jsonl", "data.json", "data.jsonl")
    assert src, "No data found. Attach your Day-2 output (train/val) or a source dataset via + Add Input."
    raw = []
    for p in src:
        raw += read_jsonl(p)
    seen, uniq = set(), []
    for r in raw:
        k = norm(r.get("instruction"))
        if k and build_answer(r).strip() and k not in seen:
            seen.add(k)
            uniq.append(r)
    random.shuffle(uniq)
    k = max(1, int(len(uniq) * 0.10))
    val_rows   = [{"messages": to_messages(r)} for r in uniq[:k]]
    train_rows = [{"messages": to_messages(r)} for r in uniq[k:]]
    print(f"Rebuilt from {len(src)} file(s): train {len(train_rows)} / val {len(val_rows)} (unique {len(uniq)})")

print("\nExample (first train item):")
for m in train_rows[0]["messages"]:
    print(f"  [{m['role']}] {m['content'][:120]}")

## Step 2 — Load the model in 4-bit and attach LoRA adapters

**What:** load `Qwen2.5-7B-Instruct` compressed to 4-bit using `bitsandbytes`, prepare it for k-bit training with PEFT, then add LoRA adapters.

**Why:** 4-bit keeps it inside 16 GB; LoRA means we train a tiny adapter instead of all 7 billion parameters.

**The settings (starting points):**
- `r = 16`, `lora_alpha = 16`, `lora_dropout = 0.05`
- adapters on the attention + MLP layers (`q, k, v, o, gate, up, down`)
- gradient checkpointing enabled to save memory

**How we check:** `print_trainable_parameters()` shows only a small % is trainable — exactly what we want.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ = 1024   # from Day-1 audit: longest example was ~896 tokens

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # Kaggle T4/P100: fp16, not bf16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device_map = {"": 0}  # one visible GPU only; avoids DataParallel + bitsandbytes 4-bit instability
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 3 — Tokenize the data and mask the questions

**What:** turn each conversation into token IDs using Qwen's chat template, then create `labels` where all tokens **before the assistant answer** are set to `-100`.

**Why:** `-100` means "ignore this token in the loss." This is how we train on the assistant's answer only, not on the system prompt or user question.

**How we check:** we print one decoded example and report how many trainable answer tokens it contains.

In [ ]:
from datasets import Dataset

ASSISTANT_MARKER = "<|im_start|>assistant\n"
assistant_marker_ids = tokenizer(ASSISTANT_MARKER, add_special_tokens=False).input_ids

def find_subsequence(sequence, subsequence):
    for i in range(0, len(sequence) - len(subsequence) + 1):
        if sequence[i:i+len(subsequence)] == subsequence:
            return i
    return -1

def tokenize_and_mask(ex):
    text = tokenizer.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=False)
    enc = tokenizer(text, truncation=True, max_length=MAX_SEQ, padding=False)
    input_ids = enc["input_ids"]
    labels = input_ids.copy()

    start = find_subsequence(input_ids, assistant_marker_ids)
    if start == -1:
        labels = [-100] * len(labels)
    else:
        answer_start = start + len(assistant_marker_ids)
        labels[:answer_start] = [-100] * answer_start

    enc["labels"] = labels
    return enc

train_ds = Dataset.from_list(train_rows).map(tokenize_and_mask, remove_columns=["messages"])
val_ds   = Dataset.from_list(val_rows).map(tokenize_and_mask, remove_columns=["messages"])

print("train:", len(train_ds), "| val:", len(val_ds))
answer_tokens = sum(x != -100 for x in train_ds[0]["labels"])
print("Answer tokens contributing to loss in first example:", answer_tokens)
print("\nDecoded first example (first 500 chars):\n")
print(tokenizer.decode(train_ds[0]["input_ids"][:180]))

## Step 4 — Configure the trainer

**What:** set the training recipe using the standard `transformers.Trainer`.

**Why each setting (starting values):**
- `learning_rate = 2e-4` — how big each learning step is (LoRA tolerates fairly high).
- effective batch = `4 × 4 = 16` — examples per update (batch × gradient accumulation).
- `QUICK_TEST = True` → **max_steps = 60** so this first run finishes in minutes and proves the pipeline works. Set it to `False` for a real 2-epoch run.
- `cosine` schedule, `paged_adamw_8bit` optimizer, `weight_decay = 0.01` — standard, memory-light, mild regularization.
- evaluate every 10 steps so we get a **validation-loss** curve.

**How we check:** the trainer builds without error and reports the effective batch size.

In [ ]:
from dataclasses import dataclass
from typing import Any
from transformers import Trainer, TrainingArguments

QUICK_TEST = True    # True = short 60-step run to prove the pipeline. Set False for a full run.

@dataclass
class CausalLMCollator:
    tokenizer: Any
    def __call__(self, features):
        labels = [f.pop("labels") for f in features]
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        max_len = batch["input_ids"].shape[1]
        padded_labels = []
        for label in labels:
            padded_labels.append(label + [-100] * (max_len - len(label)))
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

args = TrainingArguments(
    output_dir="/kaggle/working/outputs",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,          # effective batch = 2 x 8 = 16
    warmup_ratio=0.05,
    num_train_epochs=2,
    max_steps=60 if QUICK_TEST else -1,     # max_steps > 0 overrides num_train_epochs
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",
    seed=3407,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=CausalLMCollator(tokenizer),
)
print("Trainer ready. Effective batch size = 2 x 8 = 16.",
      "\nQUICK_TEST =", QUICK_TEST, "-> max_steps =", 60 if QUICK_TEST else "(full epochs)")

## Step 5 — Train, and watch for overfitting

**What:** run training and plot **train vs validation loss**.

**Why:** the loss curve is your health monitor. Good sign: both losses fall. Warning sign (**overfitting**): training loss keeps dropping while **validation loss flattens or rises** — the model is memorizing instead of learning.

**What could go wrong:**
- *Out of memory* → lower `per_device_train_batch_size` to 2 and raise `gradient_accumulation_steps` to 8 (keeps the effective batch at 16), then re-run from Step 4.

**How we check:** a loss plot is saved to `/kaggle/working/loss_curve.png`, and the validation losses are printed.

In [ ]:
stats = trainer.train()
print("\nFinished. Final train loss:", round(stats.training_loss, 4))

# Train vs validation loss (overfitting = val loss rising while train loss falls)
import matplotlib.pyplot as plt

hist = trainer.state.log_history
tr_pts = [(h["step"], h["loss"]) for h in hist if "loss" in h and "eval_loss" not in h]
ev_pts = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

if tr_pts:
    plt.figure(figsize=(7, 4))
    plt.plot(*zip(*tr_pts), label="train loss")
    if ev_pts:
        plt.plot(*zip(*ev_pts), marker="o", label="val loss")
    plt.xlabel("step"); plt.ylabel("loss"); plt.title("Training vs validation loss")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.savefig("/kaggle/working/loss_curve.png", dpi=110, bbox_inches="tight")
    plt.show()

if ev_pts:
    print("Validation loss over time:", [round(v, 4) for _, v in ev_pts])

## Step 6 — Save the adapter and sanity-check it

**What:** save the trained LoRA adapter (small — tens of MB) and ask the fine-tuned model one question to confirm it answers sensibly.

**Why:** the adapter is your deliverable. The sanity check catches obvious breakage before you rely on the model.

**How we check:** the adapter folder is written, and the model prints a coherent answer.

In [ ]:
ADAPTER = "/kaggle/working/rf_qlora_adapter"
model.save_pretrained(ADAPTER)        # saves ONLY the small LoRA adapter
tokenizer.save_pretrained(ADAPTER)
print("Adapter saved to", ADAPTER, "(this is the small file you publish later).")

# Quick sanity check: does the fine-tuned model answer sensibly?
model.eval()
msgs = [{"role": "system", "content": "You are an expert RF, DSP, and wireless communications engineer."},
        {"role": "user",   "content": "Convert 20 dBm to watts. Show the formula and the number."}]
enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                    return_tensors="pt", return_dict=True).to("cuda")
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=200, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
print("\nSample answer:\n", tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

## What you produced today
- `rf_qlora_adapter/` — your trained LoRA adapter (the thing you publish).
- `loss_curve.png` — train vs validation loss.
- A working, repeatable QLoRA pipeline.

### Record this run (experiment log)
Copy this row into your notes / `results/experiments.md` and fill the blanks:

| Exp | base | r | alpha | dropout | lr | steps | eff.batch | final train loss | best val loss | notes |
|-----|------|---|-------|---------|-----|-------|-----------|------------------|---------------|-------|
| E0  | Qwen2.5-7B | 16 | 16 | 0.05 | 2e-4 | 60 | 16 | (fill) | (fill) | first pipeline run (QUICK_TEST) |

### Save your work on Kaggle
`/kaggle/working/` is wiped at session end — click **Save Version**, or download `rf_qlora_adapter/` from the **Output** tab. (Best: publish it to the HF Hub — the optional last cell.)

### Day 4 checklist
- [ ] Data loaded/rebuilt (train + val)
- [ ] Model loaded in 4-bit + LoRA attached (small % trainable)
- [ ] Completion-only training enabled
- [ ] Short run completed and the loss looked sane
- [ ] Adapter saved + sanity answer looks reasonable
- [ ] (When ready) full run with `QUICK_TEST = False`

### Next — Day 5
Run a few **controlled experiments** (change one knob at a time: `r`, learning rate, epochs), compare **validation loss** and benchmark accuracy, and pick the best. Then re-score with the Day-3 harness to measure **before vs after**.

> **Reminder:** your 100-question benchmark still needs finishing (only 5 so far). The before/after comparison on Day 5–6 depends on it — ask me and I'll generate the full 100.

In [ ]:
# OPTIONAL — publish the adapter now (Day 7 does the full ship with a model card).
PUSH_TO_HF = False
HF_REPO = "your-username/rf-qlora-qwen2.5-7b-adapter"   # change to your HF username

if PUSH_TO_HF:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))   # add HF_TOKEN in Add-ons -> Secrets
    model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print("Pushed adapter to:", HF_REPO)
else:
    print("Skipped HF upload. To enable: set PUSH_TO_HF=True and add an HF_TOKEN secret.")